# 13 - Multi-Criteria Ranking & Semantic Deduplication

**Objective**: Compute transparent, multi-dimensional RankScores synthesizing evidence grounding, LLM judge quality, testability, rule compliance, and contradiction penalties. Perform dense semantic clustering using Sentence-Transformers to eliminate redundant propositions while preserving substantive clinical differences.

**Inputs**:
- `outputs/verification_results.json`
- `outputs/judge_results.json`
- `outputs/rule_check_results.json`
- `outputs/nli_results.json`

**Outputs**:
- `outputs/hypothesis_rankings.json`
- `outputs/deduplication_results.json`
- `outputs/final_hypotheses.csv`

**Production Code Reused**:
- `src/phase2/ranking.py`
- `src/phase2/deduplication.py`


In [ ]:
import sys, json
from pathlib import Path
import pandas as pd
import numpy as np

project_root = Path().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from src.phase2.ranking import rank_hypotheses, DEFAULT_WEIGHTS
from src.phase2.deduplication import SemanticDeduplicator

print("Ranking and deduplication engines loaded.")


## 1. Multi-Criteria Ranking Formulation
The transparent linear combination formula:
$$\text{RankScore} = w_1 \cdot S_{\text{evidence}} + w_2 \cdot S_{\text{judge}} + w_3 \cdot S_{\text{testable}} + w_4 \cdot S_{\text{rules}} - w_5 \cdot P_{\text{contra}}$$
Where $w_1=0.25, w_2=0.30, w_3=0.15, w_4=0.15, w_5=0.15$.


In [ ]:
print("Standard Configured Ranking Weights:")
for k, v in DEFAULT_WEIGHTS.items():
    print(f"  {k}: {v*100:.1f}%")

with open("outputs/hypothesis_rankings.json", "r", encoding="utf-8") as f:
    rankings_data = json.load(f)

print(f"\nTotal Ranked Hypotheses: {len(rankings_data)}")

df_ranks = pd.DataFrame(rankings_data)
display(df_ranks[["hypothesis_id", "gap_id", "rank_position", "rank_score", "quality_tier"]].head(10))


## 2. Inspect Semantic Deduplication Findings
Dense semantic embeddings (`all-MiniLM-L6-v2`) cluster candidate hypotheses. Duplicates exceeding the cosine similarity threshold (0.85) are merged into representative centroids.


In [ ]:
with open("outputs/deduplication_results.json", "r", encoding="utf-8") as f:
    dedup_data = json.load(f)

print("Deduplication Execution Trace:")
print(f"  Input Hypotheses:    {dedup_data.get('total_candidates', len(rankings_data))}")
print(f"  Retained Clusters:   {dedup_data.get('unique_clusters', len(rankings_data))}")
print(f"  Deduplication Ratio: {dedup_data.get('deduplication_ratio', 1.0):.2f}")
print(f"  Cluster Threshold:   {dedup_data.get('threshold', 0.85)}")

# Load final exported CSV
final_df = pd.read_csv("outputs/final_hypotheses.csv")
print(f"\nFinal Exported CSV Hypotheses: {len(final_df)} records")
display(final_df[["hypothesis_id", "gap_id", "rank_position", "rank_score", "target_population"]].head(5))
